# HDB Resale Flat Prices ETL Pipeline (2012-2016)
This notebook stitches together ingestion, cleaning, validation, anomaly detection, hashing, and transformation stages for the constrained dataset defined in `config/pipeline_config.yaml`. Each phase writes intermediate artifacts into their configured directories so the data lineage can be easily traced.


## Notebook Outline
1. Configuration & environment setup
2. Raw ingestion and schema harmonization
3. Cleaning, derivations, and validation
4. Anomaly detection via IQR bounds
5. Hashing + transformation
- The `resale_identifier` column fuses block digits, aggregated resale prices, the month, and the town initial into a compact key for each transaction.
- We apply SHA-256 to the identifier, ensuring the transformation is irreversible while preserving the one-to-one correspondence to the original value for uniqueness tracking.
- The hashed value is persisted in `resale_identifier_hash`, safeguarding linkage data without exposing the raw identifier.
6. Persistence of artifacts & summary statistics


In [ ]:
import hashlib
import logging
from pathlib import Path
from typing import List

import pandas as pd
import yaml

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)

config_path = Path("config/pipeline_config.yaml")
config = yaml.safe_load(config_path.read_text())

start_month = pd.to_datetime(config["start_month"], format="%Y-%m")
end_month = pd.to_datetime(config["end_month"], format="%Y-%m")
as_of_date = pd.to_datetime(config["as_of_date"])
expected_columns = config["expected_columns"]

paths = {k: Path(v) for k, v in config["paths"].items()}
raw_dir = paths["raw_data_dir"]
cleaned_dir = paths["cleaned_data_dir"]
transformed_dir = paths["transformed_data_dir"]
failed_dir = paths["failed_data_dir"]
hashed_dir = paths["hashed_data_dir"]
outputs_dir = paths["outputs_dir"]

for directory in [cleaned_dir, transformed_dir, failed_dir, hashed_dir, outputs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

logging.info("Configuration loaded: %s", config_path.resolve())
logging.info("Processing raw dir=%s, timeframe %s to %s", raw_dir, start_month.date(), end_month.date())


2026-07-20 20:47:30,035 INFO Configuration loaded: /Users/emilyng/Documents/Projects/hdb_resale_flat_prices_pipeline/config/pipeline_config.yaml
2026-07-20 20:47:30,036 INFO Processing raw dir=data/raw, timeframe 2012-01-01 to 2016-12-01


In [ ]:
def harmonize_schema(df: pd.DataFrame, expected_columns: List[str]) -> pd.DataFrame:
    """Ensure every expected column exists before downstream processing."""
    for column in expected_columns:
        if column not in df.columns:
            df[column] = pd.NA
    return df

def filter_by_month_range(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    months = pd.to_datetime(df["month"], errors="coerce", format="%Y-%m")
    mask = months.notna() & (months >= start) & (months <= end)
    filtered = df.loc[mask].copy()
    filtered["month"] = months.loc[mask]
    logging.info("Filtered %s rows between %s and %s", len(filtered), start.strftime("%Y-%m"), end.strftime("%Y-%m"))
    return filtered

def compute_remaining_lease(lease_commence_date: float, as_of: pd.Timestamp):
    """Remaining lease as of `as_of`, assuming a 99-year HDB lease, rounded down to (years, months)."""
    if pd.isna(lease_commence_date):
        return (pd.NA, pd.NA)
    lease_end = pd.Timestamp(year=int(lease_commence_date), month=1, day=1) + pd.DateOffset(years=99)
    total_months = (lease_end.year - as_of.year) * 12 + (lease_end.month - as_of.month)
    if as_of.day > lease_end.day:
        total_months -= 1
    total_months = max(total_months, 0)
    return divmod(total_months, 12)

def detect_anomalies(df: pd.DataFrame, group_by: List[str], multiplier: float) -> pd.DataFrame:
    records = []
    grouped = df.groupby(group_by, dropna=False)
    for _, group in grouped:
        q1 = group["resale_price"].quantile(0.25)
        q3 = group["resale_price"].quantile(0.75)
        if pd.isna(q1) or pd.isna(q3):
            continue
        iqr = (q3 - q1) * multiplier
        lower = q1 - iqr
        upper = q3 + iqr
        mask = (group["resale_price"] < lower) | (group["resale_price"] > upper)
        if mask.any():
            candidate = group.loc[mask].copy()
            candidate["anomaly_lower"] = lower
            candidate["anomaly_upper"] = upper
            records.append(candidate)
    if records:
        anomalies = pd.concat(records, ignore_index=True)
        logging.info("Detected %s anomalies", len(anomalies))
        return anomalies
    logging.info("No anomalies detected under current IQR bounds")
    return pd.DataFrame(columns=df.columns.tolist() + ["anomaly_lower", "anomaly_upper"])

def compute_record_hash(row: pd.Series, fields: List[str]) -> str:
    payload = "|".join(str(row.get(field, "")) for field in fields)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def extract_block_digits(block: str) -> str:
    digits_only = "".join(ch for ch in str(block or "") if ch.isdigit())
    digits_only = digits_only.zfill(3)
    return digits_only[:3]

def encode_avg_price_digits(avg_price: float) -> str:
    if pd.isna(avg_price):
        return "00"
    normalized = str(int(round(avg_price, 0)))
    normalized = normalized.zfill(2)
    return normalized[:2]

def build_resale_identifier(row: pd.Series) -> str:
    block_digits = extract_block_digits(row.get("block"))
    avg_digits = encode_avg_price_digits(row.get("avg_price"))
    month = row.get("month")
    month_digits = month.strftime("%m") if pd.notna(month) else "00"
    town_value = row.get("town", "")
    town = "" if pd.isna(town_value) else str(town_value).strip()
    town_initial = town[0].upper() if town else "X"
    return f"S{block_digits}{avg_digits}{month_digits}{town_initial}"

def hash_identifier(identifier: str) -> str:
    return hashlib.sha256(identifier.encode("utf-8")).hexdigest()

def transform_aggregates(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby(["month", "town", "flat_type"], dropna=False, as_index=False)
        .agg(
            median_price=pd.NamedAgg(column="resale_price", aggfunc="median"),
            avg_price_per_sqm=pd.NamedAgg(column="price_per_sqm", aggfunc="mean"),
            transaction_count=pd.NamedAgg(column="resale_price", aggfunc="size"),
        )
        .assign(snapshot=pd.Timestamp.utcnow().normalize())
    )
    return summary

def save_dataframe(df: pd.DataFrame, path: Path, **kwargs) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, **kwargs)
    logging.info("Persisted %s rows to %s", len(df), path)


In [3]:
raw_files = sorted(raw_dir.glob("*.csv"))
logging.info("Discovered %s raw files", len(raw_files))
frames = []
for raw_file in raw_files:
    logging.info("Loading raw source %s", raw_file.name)
    df = pd.read_csv(raw_file, dtype=str).dropna(how="all")
    df["source_file"] = raw_file.name
    df = harmonize_schema(df, expected_columns)
    frames.append(df)
if frames:
    raw_data = pd.concat(frames, ignore_index=True)
else:
    raw_data = pd.DataFrame(columns=expected_columns + ["source_file"])
logging.info("Loaded %s total rows before date filtering", len(raw_data))
raw_data = filter_by_month_range(raw_data, start_month, end_month)


2026-07-20 20:47:30,053 INFO Discovered 5 raw files
2026-07-20 20:47:30,054 INFO Loading raw source Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv
2026-07-20 20:47:30,404 INFO Loading raw source Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv
2026-07-20 20:47:30,738 INFO Loading raw source Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv
2026-07-20 20:47:30,780 INFO Loading raw source Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv
2026-07-20 20:47:30,834 INFO Loading raw source Resale flat prices based on registration date from Jan-2017 onwards.csv
2026-07-20 20:47:31,368 INFO Loaded 982011 total rows before date filtering
2026-07-20 20:47:31,445 INFO Filtered 92544 rows between 2012-01 and 2016-12


In [ ]:
cleaned = raw_data.copy()
cleaned["town"] = cleaned["town"].astype(str).str.title().str.strip()
cleaned["flat_type"] = cleaned["flat_type"].astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
cleaned["block"] = cleaned["block"].astype(str).str.upper().str.strip()
cleaned["street_name"] = cleaned["street_name"].astype(str).str.title().str.strip()
cleaned["storey_range"] = (
    cleaned["storey_range"].astype(str).str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
)
cleaned["flat_model"] = cleaned["flat_model"].fillna("Unknown").astype(str).str.title()
cleaned["floor_area_sqm"] = pd.to_numeric(cleaned["floor_area_sqm"], errors="coerce")
cleaned["resale_price"] = pd.to_numeric(cleaned["resale_price"], errors="coerce")
cleaned["lease_commence_date"] = pd.to_numeric(cleaned["lease_commence_date"], errors="coerce")
lease_components = cleaned["lease_commence_date"].apply(lambda year: compute_remaining_lease(year, as_of_date))
cleaned["remaining_lease_years"] = lease_components.apply(lambda t: t[0])
cleaned["remaining_lease_months"] = lease_components.apply(lambda t: t[1])
cleaned["remaining_lease"] = cleaned.apply(
    lambda r: f"{int(r['remaining_lease_years'])} years {int(r['remaining_lease_months'])} months"
    if pd.notna(r["remaining_lease_years"]) else pd.NA,
    axis=1,
)
cleaned["price_per_sqm"] = cleaned["resale_price"] / cleaned["floor_area_sqm"]
cleaned = cleaned.sort_values(["month", "town", "flat_type"]).reset_index(drop=True)


In [5]:
initial_row_count = len(cleaned)
essential_cols = ["month", "town", "resale_price", "floor_area_sqm"]
validation_notes = []
failed_records = []

missing_mask = cleaned[essential_cols].isna().any(axis=1)
missing_count = int(missing_mask.sum())
if missing_mask.any():
    validation_notes.append(f"{missing_count} rows dropped because of missing essential fields")
    logging.warning(validation_notes[-1])
    failed_records.append(cleaned.loc[missing_mask].copy())
    cleaned = cleaned.loc[~missing_mask].copy()

invalid_price_mask = cleaned["resale_price"] <= 0
invalid_price_count = int(invalid_price_mask.sum())
if invalid_price_mask.any():
    validation_notes.append(f"{invalid_price_count} rows with non-positive resale price detected")
    logging.warning(validation_notes[-1])
    failed_records.append(cleaned.loc[invalid_price_mask].copy())
    cleaned = cleaned.loc[~invalid_price_mask].copy()

duplicate_key = ["month", "town", "block", "street_name", "flat_type"]
pre_dedup_count = len(cleaned)
cleaned = (
    cleaned.sort_values(duplicate_key + ["resale_price"])
    .drop_duplicates(subset=duplicate_key, keep="last")
    .sort_values(["month", "town", "flat_type"])
    .reset_index(drop=True)
)
duplicates_removed = pre_dedup_count - len(cleaned)
if duplicates_removed:
    validation_notes.append(f"{duplicates_removed} duplicate transactions consolidated (kept highest price)")
    logging.info(validation_notes[-1])

agg_keys = ["month", "town", "flat_type"]
avg_price = (
    cleaned.groupby(agg_keys, dropna=False, as_index=False)["resale_price"].mean()
    .rename(columns={"resale_price": "avg_price"})
)
cleaned = cleaned.merge(avg_price, on=agg_keys, how="left")
cleaned["resale_identifier"] = cleaned.apply(build_resale_identifier, axis=1)
cleaned["resale_identifier_hash"] = cleaned["resale_identifier"].apply(hash_identifier)
cleaned.drop(columns=["avg_price"], inplace=True)
cleaned = cleaned.sort_values(["month", "town", "flat_type"]).reset_index(drop=True)

if failed_records:
    failed_df = pd.concat(failed_records, ignore_index=True)
else:
    failed_df = pd.DataFrame(columns=cleaned.columns)

failed_df = failed_df.reindex(columns=cleaned.columns)

validation_summary = {
    "processed_rows": initial_row_count,
    "remaining_rows": len(cleaned),
    "missing_essentials": missing_count,
    "invalid_price_rows": invalid_price_count,
    "duplicates_removed": duplicates_removed,
}
validation_summary


2026-07-20 20:47:31,842 INFO 8661 duplicate transactions consolidated (kept highest price)


{'processed_rows': 92544,
 'remaining_rows': 83883,
 'missing_essentials': 0,
 'invalid_price_rows': 0,
 'duplicates_removed': 8661}

In [6]:
anomalies = detect_anomalies(cleaned, config["anomaly"]["group_by"], config["anomaly"]["iqr_multiplier"])
anomalies.head()


2026-07-20 20:47:35,247 INFO Detected 2822 anomalies


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease,price_per_sqm,resale_identifier,resale_identifier_hash,anomaly_lower,anomaly_upper
0,2012-01-01,Ang Mo Kio,2 ROOM,174,Ang Mo Kio Ave 4,07 TO 09,45.0,Improved,1986,226000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN,5022.222222,S1742501A,0509ae6467b1e75c84fe77c74c7f8857f8926009d7b912...,254500.0,263300.0
1,2012-01-01,Ang Mo Kio,2 ROOM,314,Ang Mo Kio Ave 3,10 TO 12,44.0,Improved,1978,275000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN,6250.000000,S3142501A,df36be696d8b9d99bff955089fa0d1ee5cc92ae167f78a...,254500.0,263300.0
2,2012-01-01,Bedok,3 ROOM,39,Bedok Sth Rd,10 TO 12,82.0,New Generation,1978,416000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN,5073.170732,S0393401B,2f0eba762ee6c06e6b8a4fc01386b0871e54d6a2b5c57f...,268000.0,412000.0
3,2012-01-01,Bukit Merah,3 ROOM,24,Tiong Bahru Rd,01 TO 03,88.0,Standard,1973,575000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN,6534.090909,S0243801B,74e4f81c07cbc1f22d3cef3d6e351766d58b3944116eec...,294375.0,463375.0
4,2012-01-01,Bukit Merah,3 ROOM,26C,Jln Membina,16 TO 18,60.0,Model A,2009,485000.0,"Resale Flat Prices (Based on Approval Date), 2...",NaN,8083.333333,S0263801B,cd4b4ee6b32ede79c8f1bd9a48b74c47456fff34ff236d...,294375.0,463375.0


In [7]:
hash_fields = [
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "resale_price",
    "resale_identifier",
]

hashed = cleaned.copy()
hashed["resale_identifier_hash"] = hashed["resale_identifier"].apply(hash_identifier)
hashed["record_hash"] = hashed.apply(lambda row: compute_record_hash(row, hash_fields), axis=1)
transformed = transform_aggregates(hashed)
transformed.head()


,month,town,flat_type,median_price,avg_price_per_sqm,transaction_count,snapshot
0,2012-01-01,Ang Mo Kio,2 ROOM,260000.0,5763.636364,5,2026-07-20 00:00:00+00:00
1,2012-01-01,Ang Mo Kio,3 ROOM,342000.0,4905.590610,37,2026-07-20 00:00:00+00:00
2,2012-01-01,Ang Mo Kio,4 ROOM,470000.0,5031.264260,21,2026-07-20 00:00:00+00:00
3,2012-01-01,Ang Mo Kio,5 ROOM,620000.0,5477.514374,7,2026-07-20 00:00:00+00:00
4,2012-01-01,Ang Mo Kio,EXECUTIVE,730000.0,4899.328859,1,2026-07-20 00:00:00+00:00


In [8]:
failed_df = failed_df.reindex(columns=cleaned.columns)
paths_to_save = [
    (cleaned, cleaned_dir / "cleaned_resale_2012_2016.csv"),
    (hashed, hashed_dir / "hashed_resale_2012_2016.csv"),
    (transformed, transformed_dir / "resale_summary_2012_2016.csv"),
    (failed_df, failed_dir / "failed_records_2012_2016.csv"),
    (anomalies, outputs_dir / "resale_anomalies_2012_2016.csv"),
]
for dataset, target_path in paths_to_save:
    save_dataframe(dataset, target_path)


2026-07-20 20:47:36,973 INFO Persisted 83883 rows to data/cleaned/cleaned_resale_2012_2016.csv
2026-07-20 20:47:37,845 INFO Persisted 83883 rows to data/hashed/hashed_resale_2012_2016.csv
2026-07-20 20:47:37,901 INFO Persisted 6093 rows to data/transformed/resale_summary_2012_2016.csv
2026-07-20 20:47:37,902 INFO Persisted 0 rows to data/failed/failed_records_2012_2016.csv
2026-07-20 20:47:37,934 INFO Persisted 2822 rows to outputs/resale_anomalies_2012_2016.csv


In [9]:
summary = {
    "cleaned_rows": len(cleaned),
    "hashed_rows": len(hashed),
    "transformed_rows": len(transformed),
    "failed_rows": len(failed_df),
    "anomaly_rows": len(anomalies),
}
pd.Series(summary, name="count").to_frame()


,count
cleaned_rows,83883
hashed_rows,83883
transformed_rows,6093
failed_rows,0
anomaly_rows,2822


## Results & Next Steps
- Cleaned data is stored under the configured `data/cleaned` folder, hashed artifacts under `data/hashed`, and aggregates under `data/transformed`.
- Anomalous transactions detected during the IQR check are exported to `outputs/resale_anomalies_2012_2016.csv`, while integrity issues live in `data/failed` for manual review.
- Future iterations can extend this notebook with scheduling hooks, automated hash comparisons, and dashboard-ready summaries.
- Duplicate transactions now keep the higher-priced entry and feed into a single deterministic `resale_identifier` before hashing.
- The `resale_identifier` is hashed with SHA-256, providing an irreversible but unique surrogate that can be safely shared downstream.
